# GCP Cloud Run Deployment

This notebook deploys the MLOps FastAPI app to Cloud Run via the Python SDK.

**Workflow:**
1. Create Artifact Registry repository
2. Build and push Docker image
3. Deploy to Cloud Run
4. Verify endpoints

> See `../artifact_registry_cloud_run/` for the legacy CLI-based version.

In [ ]:
!pip install google-cloud-artifact-registry google-cloud-run python-dotenv requests

In [ ]:
import os
import subprocess
import requests
from dotenv import load_dotenv
from google.cloud import artifactregistry_v1
from google.cloud import run_v2

load_dotenv()

PROJECT_ID = os.environ['GCP_PROJECT_ID']
REGION = os.environ.get('GCP_REGION', 'us-central1')
GAR_REPO = os.environ.get('GAR_REPOSITORY_NAME', 'mlops-repo')
SERVICE_NAME = os.environ.get('CLOUD_RUN_SERVICE', 'mlops-api')
GCS_BUCKET = os.environ['GCS_BUCKET_NAME']
SA_EMAIL = f'mlops-least-privilege@{PROJECT_ID}.iam.gserviceaccount.com'

IMAGE_URI = f'{REGION}-docker.pkg.dev/{PROJECT_ID}/{GAR_REPO}/{SERVICE_NAME}:latest'

print(f'Project:   {PROJECT_ID}')
print(f'Region:    {REGION}')
print(f'GAR repo:  {GAR_REPO}')
print(f'Image URI: {IMAGE_URI}')

In [ ]:
# Create Artifact Registry repository (idempotent)
gar_client = artifactregistry_v1.ArtifactRegistryClient()
parent = f'projects/{PROJECT_ID}/locations/{REGION}'
repo_name = f'{parent}/repositories/{GAR_REPO}'

try:
    existing = gar_client.get_repository(name=repo_name)
    print(f'Repository already exists: {existing.name}')
except Exception:
    repo = artifactregistry_v1.Repository(
        format_=artifactregistry_v1.Repository.Format.DOCKER,
        description='MLOps Docker image repository',
    )
    op = gar_client.create_repository(
        parent=parent,
        repository_id=GAR_REPO,
        repository=repo,
    )
    result = op.result()
    print(f'Repository created: {result.name}')

## Build and Push Docker Image

Run these commands in your terminal (Docker must be running locally):

In [ ]:
print('# Step 1: Authenticate Docker with GAR')
print(f'gcloud auth configure-docker {REGION}-docker.pkg.dev')
print()
print('# Step 2: Build image')
print(f'docker build -t {SERVICE_NAME} ../06-dockerize-app-and-fastapi/')
print()
print('# Step 3: Tag')
print(f'docker tag {SERVICE_NAME} {IMAGE_URI}')
print()
print('# Step 4: Push')
print(f'docker push {IMAGE_URI}')

In [ ]:
# Deploy to Cloud Run via Python SDK
run_client = run_v2.ServicesClient()
parent_location = f'projects/{PROJECT_ID}/locations/{REGION}'
service_name_full = f'{parent_location}/services/{SERVICE_NAME}'

service = run_v2.Service(
    template=run_v2.RevisionTemplate(
        service_account=SA_EMAIL,
        containers=[
            run_v2.Container(
                image=IMAGE_URI,
                ports=[run_v2.ContainerPort(container_port=5000)],
                env=[
                    run_v2.EnvVar(name='GCS_BUCKET_NAME', value=GCS_BUCKET),
                ],
                resources=run_v2.ResourceRequirements(
                    limits={'memory': '2Gi', 'cpu': '2'},
                ),
            )
        ],
    ),
    ingress=run_v2.IngressTraffic.INGRESS_TRAFFIC_ALL,
)

try:
    existing = run_client.get_service(name=service_name_full)
    print(f'Service exists — updating...')
    service.name = service_name_full
    op = run_client.update_service(service=service)
else:
    print(f'Creating service {SERVICE_NAME}...')
    op = run_client.create_service(
        parent=parent_location,
        service=service,
        service_id=SERVICE_NAME,
    )

result = op.result()
service_url = result.uri
print(f'Service deployed: {service_url}')

In [ ]:
# Verify health endpoint
import time
time.sleep(5)  # brief wait for service to become ready

resp = requests.get(service_url, timeout=30)
print(f'Health check: {resp.status_code}')
print(resp.json())

In [ ]:
# Test pose classifier
payload = {'url': ['https://images.pexels.com/photos/1755385/pexels-photo-1755385.jpeg']}
resp = requests.post(f'{service_url}/api/v1/pose_classifier', json=payload, timeout=60)
print(f'Pose classifier: {resp.status_code}')
print(resp.json())

In [ ]:
# Teardown (uncomment to delete Cloud Run service)
# op = run_client.delete_service(name=service_name_full)
# op.result()
# print(f'Service {SERVICE_NAME} deleted.')
print('Teardown skipped. Uncomment above to delete the Cloud Run service.')